# R5.6 — Image Statistics Export

Exports **per-sample NDVI/EVI patch statistics** for the frozen binary
(coconut vs pepper) corpus so the R5.6 separability report can answer the
image-vs-tabular information question **without training any neural network**.

No model training, no architecture work, no corpus modification. Outputs
`image_stats.csv` (records joinable by `record_id`) + `image_stats_summary.json`.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = 'https://github.com/Brijesh2005/CropPrep.git'
BRANCH = 'r5.6-separability'
REPO_ROOT = Path('/kaggle/working/CropPrep')

if not (REPO_ROOT / '.git').exists():
    print(f'cloning CropPrep[{BRANCH}] -> {REPO_ROOT}')
    subprocess.run(
        ['git', 'clone', '--depth', '1', '-b', BRANCH, REPO_URL, str(REPO_ROOT)],
        check=True,
    )

sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)
print(f'repo root: {REPO_ROOT}')
print(f'cwd: {os.getcwd()}')

## 1. Environment verification

In [ ]:
import pandas as pd
import platform, sys

print('Python:', sys.version)
print('pandas:', pd.__version__)
print('platform:', platform.platform())

## 2. Frozen Data Contract Gate

Verify the frozen corpus is byte-identical before extracting anything.

In [ ]:
import csv, hashlib, json

manifest_path = REPO_ROOT / 'training_manifests' / 'crop_supervised_v2.0_manifest.json'
csv_path = REPO_ROOT / 'govt_crop_matched_v2' / 'crop_supervised_v2.csv'

def sha256(p):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for b in iter(lambda: f.read(1 << 20), b''):
            h.update(b)
    return h.hexdigest()

hashes = {
    'manifest': sha256(manifest_path),
    'csv': sha256(csv_path),
}
print(json.dumps(hashes, indent=2))
expected_manifest = '486bee061f1620872b578eee1e00e31d004e966de7e30b43bc42ebbe20dc5637'
assert hashes['manifest'] == expected_manifest, 'FROZEN MANIFEST MISMATCH'
print('frozen manifest hash OK')

## 2b. Imagery window
Same acquisition window as the R5.5 diagnostics.

In [ ]:
os.environ.setdefault('ST_IMAGERY__MODE', 'window_days')
os.environ.setdefault('ST_IMAGERY__WINDOW_DAYS', '180')
os.environ.setdefault('ST_IMAGERY__START_MONTH', '5')
os.environ.setdefault('ST_IMAGERY__SPAN_MONTHS', '12')
os.environ.setdefault('ST_IMAGERY__STRATEGY', 'closest_to_survey')
os.environ.setdefault('ST_IMAGERY__MAX_OBSERVATIONS', '8')
print('Imagery window:', os.environ['ST_IMAGERY__MODE'],
      'days=' + os.environ['ST_IMAGERY__WINDOW_DAYS'],
      'strategy=' + os.environ['ST_IMAGERY__STRATEGY'])

## 3. Bootstrap + System Check

In [ ]:
!python training/kaggle/scripts/bootstrap.py --skip-install
!python training/kaggle/scripts/system_check.py

## 4. Image statistics export (no training)

Iterates every binary sample once, extracts NDVI/EVI real-frame statistics
(Rep A nearest-frame, Rep B temporal-mean, Rep C full-sequence stats) and
writes `image_stats.csv` to `--output`.

In [ ]:
!python training/kaggle/scripts/r5_6_kaggle_image_stats.py \
    --output /kaggle/working/r5_6_image_stats \
    ; echo "r5_6_image_stats_exit_status=$?"

## 5. Verify output

In [ ]:
import json
from pathlib import Path

out = Path('/kaggle/working/r5_6_image_stats')
summary = json.loads((out / 'image_stats_summary.json').read_text(encoding='utf-8'))
print(json.dumps(summary, indent=2))
assert summary.get('verify') == 'image_stats_ok', 'IMAGE STATS ROW COUNT MISMATCH'
print('verify: image_stats_ok (10560 rows expected)')

In [ ]:
print('R5.6 image stats export completed')
print('  csv: /kaggle/working/r5_6_image_stats/image_stats.csv')
print('  summary: /kaggle/working/r5_6_image_stats/image_stats_summary.json')